# Inspeção de Qualidade de Peças de Fundição com Visão Computacional

**Mini-Projeto Avaliativo — Módulo 2 (Machine Learning e Visão Computacional)**

Este notebook implementa um pipeline que une **Visão Clássica (OpenCV)** e **Aprendizado Profundo (CNN / TensorFlow-Keras)** para inspecionar automaticamente peças de fundição metálica, classificando-as como **OK** ou **Defeituosa**.

**Etapas:**
1. Análise Exploratória (OpenCV): escala de cinza, blur, limiarização, detecção de bordas e morfologia.
2. Classificação Automatizada (CNN): ingestão em lote, Data Augmentation e treinamento.
3. Auditoria: curvas de Loss e Acurácia (Treino vs. Validação).

In [ ]:
# Bibliotecas de manipulação de dados e visualização
import os
import glob
import zipfile
import numpy as np
import matplotlib.pyplot as plt

# Visão clássica
import cv2

# Deep Learning
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

# Métricas (bônus)
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("TensorFlow:", tf.__version__)
print("OpenCV:", cv2.__version__)

## Sprint 1 — Configuração do Ambiente e Dataset

O dataset é público (*Casting Product Image Data for Quality Inspection*) e **não deve ser versionado no Git** (≈30 MB). Ele fica no Google Drive e é extraído no ambiente do Colab.

In [ ]:
# Monta o Google Drive (somente no Google Colab)
from google.colab import drive
drive.mount('/content/drive')

# Extrai o dataset (zip) do Drive para o ambiente local do Colab
caminho_zip = '/content/drive/MyDrive/casting_line_512x512.zip'
destino = '/content/dataset_pecas'

with zipfile.ZipFile(caminho_zip, 'r') as arquivo_zip:
    arquivo_zip.extractall(destino)

print("Dataset extraído em:", destino)

In [ ]:
# O image_dataset_from_directory espera uma pasta que contenha UMA SUBPASTA POR CLASSE.
# Localizamos automaticamente a pasta que contém as classes 'def_front' e 'ok_front'.
def localizar_pasta_classes(raiz):
    for caminho, subpastas, _ in os.walk(raiz):
        if any(pasta in subpastas for pasta in ('def_front', 'ok_front')):
            return caminho
    raise FileNotFoundError("Não encontrei as pastas de classe 'def_front'/'ok_front'.")

pasta_raiz = localizar_pasta_classes(destino)
print("Pasta de classes:", pasta_raiz)
print("Subpastas:", os.listdir(pasta_raiz))